In [11]:
%load_ext autoreload
%autoreload 2
import os
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"
os.environ["PMIX_MCA_gds"]="hash"

# Import useful packages
import qiskit_metal as metal
from qiskit_metal import designs, draw
from qiskit_metal import MetalGUI, Dict, open_docs
from qiskit_metal.toolbox_metal import math_and_overrides
from qiskit_metal.qlibrary.core import QComponent
from collections import OrderedDict

# To create plots after geting solution data.
import matplotlib.pyplot as plt
import numpy as np

# Packages for the simple design
from qiskit_metal.qlibrary.tlines.meandered_grounded import RouteMeanderGrounded
from qiskit_metal.qlibrary.tlines.straight_path import RouteStraight
from qiskit_metal.qlibrary.tlines.pathfinder import RoutePathfinder
from qiskit_metal.qlibrary.terminations.launchpad_wb import LaunchpadWirebond
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround
from qiskit_metal.qlibrary.terminations.short_to_ground import ShortToGround
from qiskit_metal.qlibrary.couplers.coupled_line_tee import CoupledLineTee

# Set up chip design as planar, multiplanar also available
design = designs.DesignPlanar({}, overwrite_enabled=True)

# Set up chip dimensions
design.chips.main.size.size_x = '4.8mm'
design.chips.main.size.size_y = '2.4mm'
design.chips.main.size.size_z = '500um'
design.chips.main.size.center_x = '0mm'
design.chips.main.size.center_y = '-1mm'

# Resonator and feedline gap width (W) and center conductor width (S) are set to 50 Ohm
design.variables['cpw_width'] = '10 um' #S
design.variables['cpw_gap'] = '6 um' #W

# Create GUI
gui = MetalGUI(design)

# Lauchpad 1
x1 = '-2mm'
y1 = '0mm'
launch_options1 = dict(chip='main', pos_x=x1, pos_y=y1, orientation='360', lead_length='30um', pad_height='103um',
                      pad_width='103um', pad_gap='60um')
LP1 = LaunchpadWirebond(design, 'LP1', options = launch_options1)

# Launchpad 2
x2 = '2mm'
y1 = '0mm'
launch_options2 = dict(chip='main', pos_x=x2, pos_y=y1, orientation='180', lead_length='30um', pad_height='103um',
                      pad_width='103um', pad_gap='60um')
LP2 = LaunchpadWirebond(design, 'LP2', options = launch_options2)

# Using path finder to connect the two launchpads
TL = RoutePathfinder(design, 'TL', options = dict(chip='main', trace_width ='10um',
                                            trace_gap ='6um',
                                            fillet='90um',
                                            hfss_wire_bonds = True,
                                            lead=dict(end_straight='0.1mm'),
                                            pin_inputs=Dict(
                                                start_pin=Dict(
                                                    component='LP1',
                                                    pin='tie'),
                                                end_pin=Dict(
                                                    component='LP2',
                                                    pin='tie')
                                            )))


# ######################
# # lambda/4 resonator1#
# ######################
otg1 = OpenToGround(design, 'otg1', options=dict(chip='main', pos_x='-0.2mm',  pos_y='-40um', orientation = 180))
otg2 = OpenToGround(design, 'otg2', options=dict(chip='main', pos_x='0mm',  pos_y='-1.35mm', orientation = -90))

# Use RouteMeanderGrounded to fix the total length of the resonator, and periodically
# short the center conductor to ground along its length (the "schizo" part)
res1 = RouteMeanderGrounded(design, 'resonator1',  Dict(
        trace_width ='10um',
        trace_gap ='6um',
        total_length='3.6mm',
        hfss_wire_bonds = False,
        fillet='99.9 um',
        lead = dict(start_straight='300um'),
        ground_straps=dict(
            positions=['1.2mm', '2.4mm'],
            width='10um'),
        pin_inputs=Dict(
        start_pin=Dict(component='otg1', pin='open'),
        end_pin=Dict(component='otg2', pin='open')), ))

# rebuild the GUI
gui.rebuild()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [10]:
gui.main_window.close()

True

In [9]:
x1 = '-2mm'
y1 = '0mm'
launch_options1 = dict(chip='main', pos_x=x1, pos_y=y1, orientation='360', lead_length='30um', pad_height='103um',
                      pad_width='103um', pad_gap='60um')
LP1 = LaunchpadWirebond(design, 'LP1', options = launch_options1)

# Launchpad 2
x2 = '2mm'
y1 = '0mm'
launch_options2 = dict(chip='main', pos_x=x2, pos_y=y1, orientation='180', lead_length='30um', pad_height='103um',
                      pad_width='103um', pad_gap='60um')
LP2 = LaunchpadWirebond(design, 'LP2', options = launch_options2)

# Using path finder to connect the two launchpads
TL = RoutePathfinder(design, 'TL', options = dict(chip='main', trace_width ='10um',
                                            trace_gap ='6um',
                                            fillet='90um',
                                            hfss_wire_bonds = True,
                                            lead=dict(end_straight='0.1mm'),
                                            pin_inputs=Dict(
                                                start_pin=Dict(
                                                    component='LP1',
                                                    pin='tie'),
                                                end_pin=Dict(
                                                    component='LP2',
                                                    pin='tie')
                                            )))


# ######################
# # lambda/4 resonator1#
# ######################
otg1 = OpenToGround(design, 'otg1', options=dict(chip='main', pos_x='-0.2mm',  pos_y='-40um', orientation = 180))
otg2 = OpenToGround(design, 'otg2', options=dict(chip='main', pos_x='0mm',  pos_y='-1.35mm', orientation = -90))

# Use RouteMeanderGrounded to fix the total length of the resonator, and periodically
# short the center conductor to ground along its length (the "schizo" part)
res1 = RouteMeanderGrounded(design, 'resonator1',  Dict(
        trace_width ='10um',
        trace_gap ='6um',
        total_length='3.6mm',
        hfss_wire_bonds = False,
        fillet='99.9 um',
        lead = dict(start_straight='300um'),
        ground_straps=dict(
            positions=['1.2mm', '2.4mm'],
            width='10um'),
        pin_inputs=Dict(
        start_pin=Dict(component='otg1', pin='open'),
        end_pin=Dict(component='otg2', pin='open')), ))

# rebuild the GUI
gui.rebuild()

AttributeError: 'NoneType' object has no attribute 'rebuild'

In [9]:
from SQDMetal.PALACE.Eigenmode_Simulation import PALACE_Eigenmode_Simulation
from SQDMetal.Utilities.Materials import MaterialInterface, MaterialConductor

#Eigenmode Simulation Options
user_defined_options = {
                 "mesh_refinement":  0,                             #refines mesh in PALACE - essetially divides every mesh element in half
                 "dielectric_material": "silicon",                  #choose dielectric material - 'silicon' or 'sapphire'
                 "starting_freq": 14e9,                              #starting frequency in Hz
                 "number_of_freqs": 1,                              #number of eigenmodes to find
                 "solns_to_save": 1,                                #number of electromagnetic field visualizations to save
                 "solver_order": 2,                                 #increasing solver order increases accuracy of simulation, but significantly increases sim time
                 "solver_tol": 1.0e-5,                              #error residual tolerance foriterative solver
                 "solver_maxits": 200,                              #number of solver iterations
                 "fillet_resolution":12,                            #number of vertices per quarter turn on a filleted path
                 "palace_dir":"/home/ruiz/repo/spack/opt/spack/linux-zen3/palace-develop-blpbtxe43ne2or3yo7gou6ycmdwdxuzl/bin/palace",#"PATH/TO/PALACE/BINARY",
                 "num_cpus": 8                                     #number of cpus to use in the simulation
                }

#Creat the Palace Eigenmode simulation
eigen_sim = PALACE_Eigenmode_Simulation(name ='res_eigen_test',                              #name of simulation
                                        metal_design = design,                                      #feed in qiskit metal design
                                        sim_parent_directory = "",            #choose directory where mesh file, config file and HPC batch file will be saved
                                        mode = 'simPC',                                               #choose simulation mode 'HPC' or 'simPC'
                                        meshing = 'GMSH',                                         #choose meshing 'GMSH' or 'COMSOL'
                                        user_options = user_defined_options,                        #provide options chosen above
                                        create_files = True)                                        #create mesh, config and HPC batch files

#Add in metals from layer 1 of the design file
eigen_sim.add_metallic(1)

#Add in ground plane for simulation
eigen_sim.add_ground_plane()

#Add in lumped element ports on launcher pads for 50 Ohm matching
# eigen_sim.create_port_CPW_on_Launcher('LP1', 20e-6)
# eigen_sim.create_port_CPW_on_Launcher('LP2', 20e-6)

#Fine mesh the resonator, launch apds and transmission line
eigen_sim.fine_mesh_components(['resonator1'], min_size=16e-6, max_size=100e-6, taper_dist_min=10e-6, metals_only=False)

#Sets up the lossy interfaces for MA, SA and MS interfaces
eigen_sim.setup_EPR_interfaces(metal_air=MaterialInterface('Aluminium-Vacuum'), substrate_air=MaterialInterface('Silicon-Vacuum'), substrate_metal=MaterialInterface('Silicon-Aluminium'))

#Only works in v0.14
# eigen_sim.set_farfield(ff_type='conductor', ff_material=MaterialConductor("OFHC"), ff_plane='z_neg')

#Prepares the mesh file and config file
eigen_sim.prepare_simulation()

In [10]:
#run the simulation
eigen_sim.run()

>> /usr/bin/mpirun -n 8 /home/ruiz/repo/spack/opt/spack/linux-zen3/palace-develop-blpbtxe43ne2or3yo7gou6ycmdwdxuzl/bin/palace-x86_64.bin res_eigen_test.json

_____________     _______
_____   __   \____ __   /____ ____________
____   /_/  /  __ ` /  /  __ ` /  ___/  _ \
___   _____/  /_/  /  /  /_/  /  /__/  ___/
  /__/     \___,__/__/\___,__/\_____\_____/


--> Warning!
Output folder is not empty; program will overwrite content! (outputFiles)
Git changeset ID: 3b8c020
Running with 8 MPI processes
Device configuration: cpu
Memory configuration: host-std
libCEED backend: /cpu/self/xsmm/blocked

Added 1205 elements in 1 iterations of local bisection for under-resolved interior boundaries
Added 2974 duplicate vertices for interior boundaries in the mesh
Added 7082 duplicate boundary elements for interior boundaries in the mesh
Added 1760 boundary elements for material interfaces to the mesh

Characteristic length and time scales:
 Lc = 5.760e-03 m, tc = 1.921e-02 ns
Finished partitioning 

array([[1.00000000e+00, 2.22806939e+01, 2.72306325e-05, 4.09110841e+05,
        3.06670356e-13, 3.47841803e-07],
       [2.00000000e+00, 2.23143011e+01, 2.82926168e-05, 3.94348483e+05,
        2.99543671e-13, 3.39758340e-07],
       [3.00000000e+00, 2.42029985e+01, 3.00369500e-05, 4.02887085e+05,
        9.38237284e-13, 1.06419892e-06],
       [4.00000000e+00, 2.42222326e+01, 3.00541456e-05, 4.02976564e+05,
        5.25794638e-13, 5.96384407e-07],
       [5.00000000e+00, 2.62879779e+01, 3.23508726e-05, 4.06294726e+05,
        1.17581334e-12, 1.33367094e-06]])

In [ ]:
import pyvista as pv

pvdtu = eigen_sim.retrieve_field_plots()

# The PVDVTU_Viewer wrapper (get_data_slice) only exposes a *fixed* static
# slice at z=0 rendered with matplotlib. For real interactivity, reuse its
# already-configured reader but read the full 3D field instead of slicing it.
mode_index = 0
pvdtu._reader.set_active_time_value(pvdtu._reader.time_values[mode_index])
blocks = pvdtu._reader.read()

mesh = blocks[0]
mesh['E_mag'] = np.linalg.norm(mesh['E_real'], axis=1)

# The full volume includes the simulation's air-box boundary, which has near-zero
# field and visually obscures the resonator itself. Threshold it out, keeping only
# cells where the field is actually significant (as a fraction of the peak |E|).
threshold_fraction = 0.01
mesh_thresh = mesh.threshold(threshold_fraction * mesh['E_mag'].max(), scalars='E_mag')

pl = pv.Plotter()
pl.add_mesh(mesh_thresh, scalars='E_mag', cmap='coolwarm')
pl.add_axes()
pl.show()

# If the resonator itself is still hard to see (e.g. buried inside a thicker
# high-field halo), tighten the threshold, e.g. threshold_fraction = 0.1 or 0.3

# Option A: interactive popup window with a movable clip plane through the
# (already thresholded) volume - drag it to inspect any cross-section
# pl = pv.Plotter()
# pl.add_mesh_clip_plane(mesh_thresh, scalars='E_mag', cmap='coolwarm', normal='z')
# pl.add_axes()
# pl.show()

# Option B: embed the same interactive viewer inline in the notebook instead
# of a popup window (needs: pip install trame trame-vtk trame-vuetify)
# pl = pv.Plotter(notebook=True)
# pl.add_mesh_clip_plane(mesh_thresh, scalars='E_mag', cmap='coolwarm', normal='z')
# pl.show(jupyter_backend='trame')

# Bonus: overlay vector-direction arrows (glyphs) on top of the mesh, useful to
# see field orientation, not just magnitude
# glyphs = mesh_thresh.glyph(orient='E_real', scale='E_mag', factor=2e-6, tolerance=0.02)
# pl.add_mesh(glyphs, color='black')

In [ ]:
import pandas as pd

# Palace writes eigenfrequencies + Q separately to eig.csv, one row per mode,
# with row index matching mode_index/dataset index used for the field plot above
eig_df = pd.read_csv(eigen_sim._output_data_dir + '/eig.csv')
eig_df.columns = eig_df.columns.str.strip()
print(eig_df)

f_re_col = [c for c in eig_df.columns if c.startswith('Re{f}')][0]
f_im_col = [c for c in eig_df.columns if c.startswith('Im{f}')][0]
q_col = [c for c in eig_df.columns if c.startswith('Q')][0]

row = eig_df.iloc[mode_index]
print(f"\nMode {mode_index}: f = {row[f_re_col]:.4f} GHz (Im = {row[f_im_col]:.4e} GHz), Q = {row[q_col]:.3e}")


In [4]:
from SQDMetal.PALACE.Frequency_Driven_Simulation import PALACE_Driven_Simulation

# Driven (S-parameter) simulation options
driven_options = {
                 "dielectric_material": "silicon",
                 "solns_to_save": 4,                                #number of field snapshots saved across the sweep, for visualization
                 "solver_order": 2,
                 "solver_tol": 1.0e-7,
                 "solver_maxits": 200,
                 "solver_initial_guess": True,                      #reuse each frequency step's solution as the next step's initial guess
                 "fillet_resolution": 12,
                 "palace_dir": "/home/ruiz/repo/spack/opt/spack/linux-zen3/palace-develop-blpbtxe43ne2or3yo7gou6ycmdwdxuzl/bin/palace",
                 "num_cpus": 8
                }

driven_sim = PALACE_Driven_Simulation(name='res_driven_test',
                                       metal_design=design,
                                       sim_parent_directory="",
                                       mode='simPC',
                                       meshing='GMSH',
                                       user_options=driven_options,
                                       create_files=True)

driven_sim.add_metallic(1)
driven_sim.add_ground_plane()

# Two 50-Ohm ports on the launchpads: drive at LP1, measure transmission at LP2
driven_sim.create_port_CPW_on_Launcher('LP1', 20e-6)
driven_sim.create_port_CPW_on_Launcher('LP2', 20e-6)
driven_sim.set_port_excitation(1)  # excite port 1 (LP1); LP2 is the passive/measurement port

driven_sim.fine_mesh_components(['TL', 'resonator1', 'LP1', 'LP2'], min_size=16e-6, max_size=100e-6, taper_dist_min=10e-6, metals_only=False)

# Sweep bracketing the eigenmode cluster found earlier (~22-24GHz). 400 points here -
# tune freq_step for less runtime (coarser) or better resolution of the near-degenerate
# pair (finer), since each point is a separate full-wave solve.
driven_sim.set_freq_values(freq_start=20e9, freq_end=26e9, freq_step=50e6)

driven_sim.prepare_simulation()

Error   : Gmsh has not been initialized
Error   : Gmsh has not been initialized


In [5]:
#run the driven (S-parameter) sweep
driven_sim.run()

>> /usr/bin/mpirun -n 8 /home/ruiz/repo/spack/opt/spack/linux-zen3/palace-develop-blpbtxe43ne2or3yo7gou6ycmdwdxuzl/bin/palace-x86_64.bin res_driven_test.json

_____________     _______
_____   __   \____ __   /____ ____________
____   /_/  /  __ ` /  /  __ ` /  ___/  _ \
___   _____/  /_/  /  /  /_/  /  /__/  ___/
  /__/     \___,__/__/\___,__/\_____\_____/


--> Warning!
Output folder is not empty; program will overwrite content! (outputFiles)
Git changeset ID: 3b8c020
Running with 8 MPI processes
Device configuration: cpu
Memory configuration: host-std
libCEED backend: /cpu/self/xsmm/blocked

Added 2450 elements in 2 iterations of local bisection for under-resolved interior boundaries
Added 5048 duplicate vertices for interior boundaries in the mesh
Added 12408 duplicate boundary elements for interior boundaries in the mesh
Added 1859 boundary elements for material interfaces to the mesh

Characteristic length and time scales:
 Lc = 5.760e-03 m, tc = 1.921e-02 ns
Finished partitionin

{'freqs': array([2.000e+10, 2.005e+10, 2.010e+10, 2.015e+10, 2.020e+10, 2.025e+10,
        2.030e+10, 2.035e+10, 2.040e+10, 2.045e+10, 2.050e+10, 2.055e+10,
        2.060e+10, 2.065e+10, 2.070e+10, 2.075e+10, 2.080e+10, 2.085e+10,
        2.090e+10, 2.095e+10, 2.100e+10, 2.105e+10, 2.110e+10, 2.115e+10,
        2.120e+10, 2.125e+10, 2.130e+10, 2.135e+10, 2.140e+10, 2.145e+10,
        2.150e+10, 2.155e+10, 2.160e+10, 2.165e+10, 2.170e+10, 2.175e+10,
        2.180e+10, 2.185e+10, 2.190e+10, 2.195e+10, 2.200e+10, 2.205e+10,
        2.210e+10, 2.215e+10, 2.220e+10, 2.225e+10, 2.230e+10, 2.235e+10,
        2.240e+10, 2.245e+10, 2.250e+10, 2.255e+10, 2.260e+10, 2.265e+10,
        2.270e+10, 2.275e+10, 2.280e+10, 2.285e+10, 2.290e+10, 2.295e+10,
        2.300e+10, 2.305e+10, 2.310e+10, 2.315e+10, 2.320e+10, 2.325e+10,
        2.330e+10, 2.335e+10, 2.340e+10, 2.345e+10, 2.350e+10, 2.355e+10,
        2.360e+10, 2.365e+10, 2.370e+10, 2.375e+10, 2.380e+10, 2.385e+10,
        2.390e+10, 2.395e+10,

In [6]:
s_data = driven_sim.retrieve_data()

freqs_GHz = s_data['freqs'] / 1e9
S21_dB = 20 * np.log10(np.abs(s_data['S21']))
S11_dB = 20 * np.log10(np.abs(s_data['S11']))

fig, axs = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
axs[0].plot(freqs_GHz, S21_dB)
axs[0].set_ylabel('|S21| (dB)')
axs[0].set_title('Transmission / reflection spectrum')
axs[0].grid(True)

axs[1].plot(freqs_GHz, S11_dB, color='C1')
axs[1].set_xlabel('Frequency (GHz)')
axs[1].set_ylabel('|S11| (dB)')
axs[1].grid(True)

fig.tight_layout()

# The near-degenerate mode pair (~22GHz) should show up as two closely-spaced
# dips in |S21| (or peaks in |S11|), cross-checking the eigenmode results.